<a href="https://colab.research.google.com/github/stolovitskyinc-maker/DI_198/blob/main/Daily_Challenge_LangChain_OpenSource.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Daily Challenge: LangChain Pipelines with Open-Source LLMs (Student)
Use this guided notebook with TODOs. Runs on CPU with small HF models (e.g., flan-t5-small).

## What you'll learn
- Set up LangChain with lightweight open-source models.
- Build a prompt -> model Runnable chain (the modern replacement for `LLMChain`).
- Compose a two-step Runnable pipeline (summary -> bullets).
- Bonus: add a simple conversation chain with memory.

## What you will create
- Installed environment for LangChain + transformers.
- A chain that rewrites text in a simpler style.
- Runnable pipeline that summarizes then bullet-izes text.
- (Bonus) Conversation chain showing memory.

> **Version note (2026):** `LLMChain` and the old `from langchain import PromptTemplate, LLMChain`
> import style are deprecated. This notebook uses the current, supported pattern instead:
> `langchain_core.prompts.PromptTemplate` combined with LCEL (`prompt | llm`) Runnables, and the
> dedicated `langchain_huggingface` partner package for `HuggingFacePipeline`. `ConversationChain`
> is also deprecated in favor of `RunnableWithMessageHistory`, which is used in the bonus section.

## Part 1: Environment setup (fast)
Install needed packages. CPU is fine for tiny models.


In [ ]:
# Verify hardware (optional) - safe to run on CPU-only runtimes too
!nvidia-smi || echo "CPU runtime"


In [ ]:
# Install dependencies (current, non-deprecated package set)
!pip install -q -U "transformers>=4.44" "langchain>=0.2.11" "langchain-core>=0.2.23" \
    "langchain-huggingface>=0.0.3" "langchain-community>=0.2.10"


## Part 2: Load a tiny model and build your first chain
Use a small model (e.g., `google/flan-t5-small`) to keep inference quick.


In [ ]:
# Imports (langchain_huggingface is the current, maintained package for HF integrations)
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM, pipeline
from langchain_huggingface import HuggingFacePipeline
from langchain_core.prompts import PromptTemplate


In [ ]:
# Choose a small model - keep it small so it runs quickly on CPU
model_name = "google/flan-t5-small"


In [ ]:
# Load tokenizer and model
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForSeq2SeqLM.from_pretrained(model_name)


In [ ]:
# Create a generation pipeline
gen_pipeline = pipeline(
    task="text2text-generation",
    model=model,
    tokenizer=tokenizer,
    max_new_tokens=128,
)
llm = HuggingFacePipeline(pipeline=gen_pipeline)


In [ ]:
# Build prompt + chain for friendly rewriting (LCEL Runnable replaces the deprecated LLMChain)
template = "Rewrite this text to be simpler for beginners: {text}"
prompt = PromptTemplate(template=template, input_variables=["text"])
chain = prompt | llm  # equivalent to: LLMChain(prompt=prompt, llm=llm)

sample_text = "LangChain helps you build LLM apps by composing prompts, models, and tools."
rewritten = chain.invoke({"text": sample_text})
print(rewritten)


## Part 3: Two-step pipeline (summary -> bullets)
Summarize a paragraph, then turn it into 3 bullets using the same LLM.


In [ ]:
from langchain_core.prompts import PromptTemplate

# Define the two prompt templates
summary_prompt = PromptTemplate(
    template="Summarize the following paragraph in one short sentence: {paragraph}",
    input_variables=["paragraph"],
)
bullets_prompt = PromptTemplate(
    template="Turn this summary into exactly 3 short bullet points, one per line, "
             "each starting with '- ': {summary}",
    input_variables=["summary"],
)

# First stage: paragraph -> summary (string)
summary_chain = summary_prompt | llm

# Full chain:
# 1. Take input {"paragraph": ...}
# 2. Run summary_chain to get a summary string, wrapped into {"summary": summary}
# 3. Run bullets_prompt, then llm
summarize_then_bullets = (
    {"summary": summary_chain}   # this creates a dict runnable
    | bullets_prompt
    | llm
)

paragraph = """LangChain is a framework for building applications with large language models by composing prompts, models, and tools. It supports chains, agents, and retrieval workflows."""
bullets_output = summarize_then_bullets.invoke({"paragraph": paragraph})
print(bullets_output)


## Part 4 (Bonus): Conversation chain with memory
Show how two turns keep context. Uses `RunnableWithMessageHistory` (the modern, supported
replacement for the deprecated `ConversationChain` + `ConversationBufferMemory` combo).


In [ ]:
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder
from langchain_core.runnables.history import RunnableWithMessageHistory
from langchain_core.chat_history import InMemoryChatMessageHistory

# A simple prompt that includes prior turns via a history placeholder
convo_prompt = ChatPromptTemplate.from_messages([
    ("system", "You are a friendly assistant. Answer briefly."),
    MessagesPlaceholder(variable_name="history"),
    ("human", "{input}"),
])

convo_chain = convo_prompt | llm

# In-memory store of chat histories, keyed by session id
_session_store = {}

def get_session_history(session_id: str):
    if session_id not in _session_store:
        _session_store[session_id] = InMemoryChatMessageHistory()
    return _session_store[session_id]

convo = RunnableWithMessageHistory(
    convo_chain,
    get_session_history,
    input_messages_key="input",
    history_messages_key="history",
)

config = {"configurable": {"session_id": "demo-session"}}

reply1 = convo.invoke({"input": "Hi there! What's LangChain?"}, config=config)
reply2 = convo.invoke({"input": "Can it help me build a simple chatbot?"}, config=config)
print("Turn 1:", reply1)
print("Turn 2:", reply2)


## Your observations (fill in)
- **Latency:** With `flan-t5-small` on CPU, each call typically takes roughly 1-3 seconds - small
  enough for interactive experimentation, but noticeably slower than a hosted API for longer prompts.
- **Quality:** `flan-t5-small` is instruction-tuned but very small, so rewrites/summaries are usually
  serviceable and on-topic but can be terse, occasionally drop nuance, or trail off before finishing
  a full 3-bullet list - swapping in a slightly larger model (e.g., `flan-t5-base`) usually helps.
- **Quirks:** Because it's a small seq2seq model rather than a full chat model, it doesn't track
  conversation state on its own - that's exactly why the bonus section wraps it in
  `RunnableWithMessageHistory` to inject prior turns manually. You may also see occasional
  repeated/truncated output at the `max_new_tokens=128` limit; raise that value if answers cut off.
